In [169]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import math
from tqdm import tqdm

In [170]:
#hyper_parameters
num_heads=8
d_model=512
#模型内部使用的三个超参数，对外只有d_model
d_k=64
d_v=64 #保证d_v*num_head=d_model
d_ff=2048
lr=1e-4
epochs=1001
device='cuda:0'

# 先写整体架构再对其输入

### 1.加mask的attention模块计算

In [171]:
def Attention(Q,K,V):
#   alpha=torch.softmax((Q@K.transpose(-2,-1))/(d_k**0.5),dim=-1)#需要指定转置（qkv均为三维（含batch）），需要指定按行softmax
#加上casule mask:
    B,H,N,_=Q.shape
    mask=torch.tril(torch.ones(B,H,N,N))#tril生成下三角
    #再将所有0位置替换为布尔无穷小
    mask=mask.masked_fill(mask == 0, float('-inf'))
    #再将所有1替换为布尔0
    mask=mask.masked_fill(mask==1,0.0)
    mask=mask.to(Q.device)
    alpha=torch.softmax((Q@K.transpose(-2,-1))/(d_k**0.5)+mask,dim=-1)

    h=alpha@V
    return h

### 2.位置编码模块

In [172]:

class PositionalEmbed(nn.Module):
    def __init__(self,d_model,pos,base=10000,dropout=0.1):#embedding 后的x维度恰好是d_model，pos为序列最大长度（也就是每句tokens长度）
        super().__init__()
        self.dropout=nn.Dropout(p=dropout)#Dropout模块
        
        #position向量：
        p=torch.arange(0,pos,dtype=torch.float) #保证是float向量
        
        #角度向量:先生成等差的指数部分，再整体exp，否则无法直接生成
        factor=torch.arange(0,(d_model+1)//2,dtype=torch.float)
        factor=(-2/d_model*math.log(base))*factor
        factor=torch.exp(factor)

        angle=torch.matmul(p.unsqueeze(1),factor.unsqueeze(0))#torch的一维向量没有行列概念，所有一维向量相乘都只有内积，所以需要使用unsqueeze函数
        #（pos,）是一维，pytorch中一维向量没有行列之分，（pos,1）就是二维了（列矩阵）
        #matmul对一维向量执行点积，对二维向量执行矩阵乘法
        #可以使用 .reshape(pos, 1) 或 .view(pos, 1) 将一维张量 (pos,) 转换为二维列向量 (pos, 1)。这两种方法与 .unsqueeze(1) 的效果等价，都是增加一个维度。
        
        pe=torch.zeros(pos,d_model)
        pe[:,0::2]=torch.sin(angle)#0::2表示从0开始每隔两个取一个    #对整个e做出预修改然后取用
        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(angle)
        else:
            pe[:, 1::2] = torch.cos(angle[:, :d_model//2]) 

        self.register_buffer('pe',pe)#将init中的局部变量e变为全局

        
    def forward(self,x):#forward函数负责将计算好的位置编码加到embedding(x)上,
                          #因为nn.Module机制规定只有forward函数里的操作才能参与反向传播，追踪梯度
        x=x+self.pe[:x.size(1)]#为了追踪这个加法的梯度
        x=self.dropout(x)
        return x   


### 3.嵌入层

In [173]:
class Embedding(nn.Module):
    def __init__(self,num_embeddings,d_model):
        super().__init__()
        self.embedding=nn.Embedding(num_embeddings=num_embeddings,embedding_dim=d_model) 
    def forward(self,x):
        x=self.embedding(x)
        return x

### 4.泛化的多头注意力

In [174]:
class Multihead(nn.Module):
    def __init__(self,d_model,d_k,d_v):
        super().__init__() 
        

        #多头注意力：[W1|W2|W3|W4|...]
        self.w_q=nn.Linear(d_model,d_k*num_heads) 
        self.w_k=nn.Linear(d_model,d_k*num_heads)
        self.w_v=nn.Linear(d_model,d_v*num_heads)  
        self.w0=nn.Linear(d_v*num_heads,d_model)
        #self.output_proj=nn.Linear(d_model,vocab_size+1)#transformer中不需要映射到词表大小，只需要保持d_model
    
    
    def forward(self,xq,xk,xv):#有可能是交叉注意力模块
        
        
        Q=self.w_q(xq)
        K=self.w_k(xk)
        V=self.w_v(xv)
        #先计算后拆分
        Bq,Lq,numxKq=Q.shape#K与Q形状完全一致
        Bv,Lv,numxKv=V.shape
        Q=Q.reshape(Bq,Lq,num_heads,numxKq//num_heads)#reshape都是重排元素，所以只能拆为相邻元素
        K=K.reshape(Bq,Lq,num_heads,numxKq//num_heads)
        V=V.reshape(Bv,Lv,num_heads,numxKv//num_heads)
        Q = Q.transpose(1, 2)  # (B, L,num_heads, d_k) -> (B, num_heads, L, d_k)   转置第1，2维度
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        h_fin = Attention(Q, K, V)  # 输出 (B, Heads, L, d_v)
        h_fin = h_fin.transpose(1,2)  # (B, L, H, d_v)
        h_fin=h_fin.flatten(start_dim=2)  #(B,L,H*d_v)
        h_fin=self.w0(h_fin)


        #logits = self.output_proj(h_fin)  #出错，输出应该还是d_model

        return h_fin





### 5.encoder 模块
N个串行的encoder模块，最后一个块把输出给decoder第一个块

In [175]:
class Encoder_block(nn.Module):
    def __init__(self,d_model,d_k,d_v,d_ff=2048):
        super().__init__()
        self.multihead=Multihead(d_model,d_k,d_v)
        self.LN1=nn.LayerNorm(d_model)
        self.LN2=nn.LayerNorm(d_model)#如果都用一套layernorm的话，参数复用
        self.dropout=nn.Dropout(p=0.1)
        self.ffn=nn.Sequential(
                nn.Linear(num_heads*d_v,d_ff), # num_heads*d_v=d_model
                nn.ReLU(),
                nn.Linear(d_ff,d_model)  #将输出维度投影回d_model维度，给下一个块使用
        )

    def forward(self,x):
        y1=self.multihead(xq=x,xk=x,xv=x)
        y1=self.dropout(y1)
        #print('y1shape:',y1.shape)  #B，L，151   #151是因为多头注意力没删最后映射到词表的旧模板
        #print('xshape',x.shape)   #B,L,512
        x1=x+y1
        x2=self.LN1(x1)
       
        
        y2=self.ffn(x2)
        y2=self.dropout(y2)
        x3=x2+y2
        x3=self.LN2(x3)
        return x3
        

# Encoder最终实现

In [176]:
class Encoder(nn.Module):
    def __init__(self,d_model,d_k,d_v,d_ff=2048,block_num=6):
        super().__init__()
        #多层block堆叠写法：
        for i in range(block_num):
            self.encoder=nn.ModuleList([
                Encoder_block(d_model,d_k,d_v) for _ in range(block_num)
            ])#使用列表动态规定encoder_block块数量
    def forward(self,x):
        for block in self.encoder:
            x=block(x) #可以实现串行
        return x 

### 7.decoder的输入问题？
在训练时：decoder的输入就是已知的目标序列——————teacher forcing  
在推理时：输入是已经生成的tokens的自回归  
encoder输出形成的QK是作为间接输入给encoder的  
#### *teacher forcing:
训练时直接将目标序列右移一位输入给decoder  
目标序列 我 爱 猫 为例，实际喂给 Decoder 的是错开一位的序列：  
Decoder 输入	  Decoder 应输出  
<BOS> 我 爱	    我 爱 猫 <EOS>  
可以实现同时对三个位置进行并行训练，高效




### 8.decoder模块
相比于encoder中间多一层交叉注意力

In [177]:
class Decoder_block(nn.Module):
    def __init__(self,d_model,d_k,d_v,d_ff=2048):
        super().__init__()
        self.multihead=Multihead(d_model,d_k,d_v)
        self.LN1=nn.LayerNorm(d_model)
        self.LN2=nn.LayerNorm(d_model)#如果都用一套layernorm的话，参数复用
        self.dropout=nn.Dropout(p=0.1)
        self.ffn=nn.Sequential(
                nn.Linear(num_heads*d_v,d_ff), # num_heads*d_v=d_model
                nn.ReLU(),
                nn.Linear(d_ff,d_model)  #将输出维度投影回d_model维度，给下一个块使用
        )

    def forward(self,x,h):#h为最后一个encoder块的输出状态
        y1=self.multihead(xq=x,xk=x,xv=x)
        y1=self.dropout(y1)
        x1=x+y1
        x2=self.LN1(x1)
        #交叉注意力：每一个decoder块都接收最后一个encoder块的输出作为交叉注意力QK来源
        y2=self.multihead(xq=x2,xk=h,xv=h)
        y2=self.dropout(y2)
        x3=x2+y2
        
        
        y3=self.ffn(x3)
        y3=self.dropout(y3)
        x4=x3+y3
        x4=self.LN2(x4)
        return x4
        

# decoder最终实现

In [178]:
class Decoder(nn.Module):
    def __init__(self,d_model,d_k,d_v,d_ff=2048,block_num=6):
        super().__init__()
        #多层block堆叠写法：
        for i in range(block_num):
            self.encoder=nn.ModuleList([
                Decoder_block(d_model,d_k,d_v) for _ in range(block_num)
            ])#使用列表动态规定encoder_block块数量
    def forward(self,x,h):
        for block in self.encoder:
            x=block(x,h) #可以实现串行
        return x 

### 10 decoder输出处理模块

In [179]:
class Output_operation(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.linear=nn.Linear(input_dim,output_dim)
    def forward(self,x):
        x=self.linear(x)
        return x

### 11.开始组合,处理输入
### 原始的x_train给encoder输入，右移一位的y_train给decoder

In [180]:
#原始的x_train给encoder输入，右移一位的y_train给decoder
data_path='./data/poem.txt'
data=pd.read_csv(data_path,header=None,names=['text'])
data['tokens']=data['text'].str.split()
#中文：data['tokens'] = data['text'].apply(lambda x: list(str(x)))

data.shape


(20, 2)

In [181]:
#tockenizer:
all_words=[word for tokens in data['tokens'] for word in tokens]
vocab={word:idx for idx,word in enumerate(set(all_words),start=1)}
vocab_size=len(set(all_words))
print('vocabulary:',vocab)
data['tokens_ids']=data['tokens'].apply(lambda x: [vocab[word] for word in x])

data.size

vocabulary: {'寄': 1, '雨': 2, '画': 3, '湿': 4, '青': 5, '桂': 6, '洲': 7, '独': 8, '尽': 9, '畔': 10, '淡': 11, '亭': 12, '草': 13, '处': 14, '送': 15, '帷': 16, '晚': 17, '里': 18, '又': 19, '生': 20, '入': 21, '随': 22, '桃': 23, '上': 24, '桥': 25, '吹': 26, '如': 27, '庭': 28, '月': 29, '舟': 30, '山': 31, '胧': 32, '江': 33, '阑': 34, '远': 35, '不': 36, '溪': 37, '去': 38, '深': 39, '见': 40, '绕': 41, '春': 42, '白': 43, '声': 44, '疏': 45, '谁': 46, '有': 47, '红': 48, '波': 49, '秋': 50, '玉': 51, '帘': 52, '归': 53, '孤': 54, '何': 55, '家': 56, '风': 57, '难': 58, '灯': 59, '黄': 60, '<end>': 61, '落': 62, '烟': 63, '倚': 64, '渐': 65, '楼': 66, '年': 67, '水': 68, '到': 69, '藕': 70, '寻': 71, '逢': 72, '絮': 73, '残': 74, '干': 75, '把': 76, '暮': 77, '千': 78, '无': 79, '枕': 80, '载': 81, '影': 82, '梦': 83, '漠': 84, '东': 85, '故': 86, '柳': 87, '香': 88, '空': 89, '晓': 90, '襟': 91, '长': 92, '灭': 93, '方': 94, '向': 95, '边': 96, '隐': 97, '蓝': 98, '迢': 99, '星': 100, '雁': 101, '悄': 102, '凋': 103, '微': 104, '苍': 105, '轻': 106, '事': 107, '离': 108, '南': 109, '

60

In [182]:
input_ids=np.array(data['tokens_ids'].tolist())

batch_size,_=input_ids.shape
input_ids.shape,batch_size


((20, 15), 20)

In [183]:
x=torch.tensor(input_ids)
x.shape

torch.Size([20, 15])

In [184]:
#数据准备
x_train=x[:,:-1]#给encoder
y_train=x[:,1:] #给decoder
print("x_train shape:", x_train.shape)  
print("y_train shape:", y_train.shape)
x_train.dtype

x_train shape: torch.Size([20, 14])
y_train shape: torch.Size([20, 14])


torch.int64

## bug1:预处理封装为类，只随机化一次

In [185]:
pos=x_train.shape[1]
#数据预处理：embedding与positionalembedding
class Input_operation(nn.Module):
    def __init__(self,num_embeddings=vocab_size+1,d_model=d_model,max_len=pos):
        super().__init__()
        self.embed_model=Embedding(num_embeddings=vocab_size+1,d_model=d_model)
        self.positionalembed_model=PositionalEmbed(d_model,max_len)
    def forward(self,x):
        pos=x.shape[1]
        x_stand=self.positionalembed_model(self.embed_model(x))
    
        return x_stand
        
#def pre_operation(x):
#    pos=x.shape[1]
#    embed_model=Embedding(num_embeddings=vocab_size+1,d_model=d_model)
#    embed_model=embed_model.to(device)
#    positionalembed_model=PositionalEmbed(d_model,pos)
#    positionalembed_model=positionalembed_model.to(device)
#    
#    x_stand=positionalembed_model(embed_model(x))    
#    return x_stand

## 封装为一个transformer类的写法：

In [190]:

class Transformer(nn.Module):
    def __init__(self,d_model,d_k,d_v,d_ff,vocab_size):
        super().__init__()

        self.encoder=Encoder(d_model,d_k,d_v,d_ff)
 
        self.decoder=Decoder(d_model,d_k,d_v,d_ff)

    def forward(self,x,y):
        x_h=self.encoder(x)#会输出状态
        y_h=self.decoder(y,x_h)#将输出的x状态再输入
        return y_h


model=Transformer(d_model,d_k,d_v,d_ff,vocab_size)
model.to(device)



input_operation=Input_operation(vocab_size+1,d_model)
input_operation=input_operation.to(device)
output_operation=Output_operation(d_model,vocab_size+1)
output_operation=output_operation.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(list(model.parameters())+list(input_operation.parameters())+list(output_operation.parameters()),lr=lr)


for epoch in tqdm(range(epochs)):
    x_train=x_train.to(device)
    y_train=y_train.to(device)
    #y预处理：embed,positionembed
    x_in=input_operation(x_train)
    y_in=input_operation(y_train)
    y_pred=model(x_in,y_in)
     
    y_pred=y_pred.transpose(1,2)
    y_train=y_train.squeeze(-1)
    loss=criterion(y_pred, y_train) 
    # 3. 反向传播 + 更新参数
    optimizer.zero_grad()   # 清空上一步的梯度
    loss.backward()         # 计算梯度
    optimizer.step()        # 更新参数
    
    if (epoch + 1) % 1000 == 0:
        print(f"epoch {epoch+1}, loss: {loss.item():.4f}")

100%|██████████| 1001/1001 [01:34<00:00, 10.54it/s]

epoch 1000, loss: 0.0008


## 非封装Transformer写法

In [186]:
#模型定义
input_operation=Input_operation(vocab_size+1,d_model)
input_operation=input_operation.to(device)
encoder=Encoder(d_model,d_k,d_v,d_ff)
encoder=encoder.to(device)
decoder=Decoder(d_model,d_k,d_v,d_ff)
decoder=decoder.to(device)
output_operation=Output_operation(d_model,vocab_size+1)
output_operation=output_operation.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    list(input_operation.parameters())+
    list(encoder.parameters()) + 
    list(decoder.parameters()) + 
    list(output_operation.parameters()), 
    lr=lr
) #三个模块的都要计算梯度！！！


for epoch in tqdm(range(epochs)):
    x_train=x_train.to(device)
    y_train=y_train.to(device)
    #y预处理：embed,positionembed
    x_in=input_operation(x_train)
    y_in=input_operation(y_train)
    
    #输入
    #print("x_h shape",x_in.shape)
    x_h=encoder(x_in)#会输出状态
    y_h=decoder(y_in,x_h)#将输出的x状态再输入
    y_pred=output_operation(y_h)
    
    #计算loss
    y_pred=y_pred.transpose(1,2)
    y_in=y_in.squeeze(-1)
    loss=criterion(y_pred, y_train) 
    # 3. 反向传播 + 更新参数
    optimizer.zero_grad()   # 清空上一步的梯度
    loss.backward()         # 计算梯度
    optimizer.step()        # 更新参数
    
    if (epoch + 1) % 1000 == 0:
        print(f"epoch {epoch+1}, loss: {loss.item():.4f}")

100%|██████████| 1001/1001 [01:34<00:00, 10.59it/s]

epoch 1000, loss: 0.0008


In [192]:
def generate_text(model, input_operation, output_operation, src_tokens, 
                  start_id, end_id, max_len=50):
    model.eval()
    with torch.no_grad():
        # src_tokens: 1D tensor of token ids, shape (L,)
        src = src_tokens.unsqueeze(0).to(device)          # (1, L)
        src_emb = input_operation(src)                    # (1, L, d_model)
        memory = model.encoder(src_emb)                  # (1, L, d_model)

        # 目标序列初始为 start_id
        tgt = torch.tensor([[start_id]], device=device)   # (1, 1)
        for _ in range(max_len):
            tgt_emb = input_operation(tgt)                # (1, tgt_len, d_model)
            dec_out = model.decoder(tgt_emb, memory)      # (1, tgt_len, d_model)
            logits = output_operation(dec_out)            # (1, tgt_len, vocab_size)
            next_logits = logits[:, -1, :]                # (1, vocab_size)
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)  # (1,1)
            if next_token.item() == end_id:
                break
            tgt = torch.cat([tgt, next_token], dim=1)    # 拼接
        return tgt.squeeze(0).cpu().tolist()              # 返回 token id 列表

# 示例调用（使用你数据中的某个样本）
sample_src = x_train[0]                # 取第一句作为源
# 假设你定义 start_id = vocab['<bos>'] 或任意固定值（如诗句的第一个字）
# 若没有 BOS，可先取源序列的第一个词作为起始，但这样会重复。
# 更常用的是固定一个起始词（如 '春'）并让模型续写。
tokens_generated = generate_text(model, input_operation, output_operation, 
                                 sample_src, start_id=vocab['春'], end_id=vocab['<end>'])
generated_sentence = ' '.join([idx_to_word[id] for id in tokens_generated])
print(generated_sentence)

RuntimeError: shape '[1, 1, 8, 64]' is invalid for input of size 7168